# Exploratory Data Analysis (EDA) - PaySim

In this notebook, we analyze the PaySim dataset to understand the behavioral patterns of fraud. We will look at transaction types, amounts, and correlations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")

# Set pandas display options to avoid scientific notation for readability
pd.set_option('display.float_format', lambda x: '%.3f' % x)

### 1. Load the Dataset

In [ ]:
data_path = '../data/raw/PS_20174392719_1491204439457_log.csv'

try:
    df = pd.read_csv(data_path)
    print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns.")
except FileNotFoundError:
    print("Error: Dataset not found. Please ensure it is in the data/raw/ directory.")

### 2. Transaction Types and Fraud Distribution
Fraud typically only happens in specific transaction types (like TRANSFER and CASH_OUT). Let's verify this.

In [ ]:
if 'df' in locals():
    plt.figure(figsize=(10, 6))
    ax = sns.countplot(data=df, x='type', hue='isFraud')
    plt.title('Transaction Types: Legitimate vs Fraudulent')
    plt.yscale('log') # Log scale because legitimate transactions dwarf fraud
    plt.ylabel('Count (Log Scale)')
    plt.show()
    
    print("Fraud counts by transaction type:")
    display(df[df['isFraud'] == 1]['type'].value_counts())

### 3. Transaction Amount Analysis
Are fraudulent transactions significantly larger or smaller than legitimate ones?

In [ ]:
if 'df' in locals():
    # Filter to only TRANSFER and CASH_OUT since they contain the fraud
    df_filtered = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])]
    
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df_filtered, x='isFraud', y='amount')
    plt.title('Transaction Amount Distribution: Legitimate vs Fraud')
    plt.yscale('log')
    plt.ylabel('Amount (Log Scale)')
    plt.show()
    
    print("Summary statistics for Fraudulent amounts:")
    display(df_filtered[df_filtered['isFraud'] == 1]['amount'].describe())
    print("\nSummary statistics for Legitimate amounts:")
    display(df_filtered[df_filtered['isFraud'] == 0]['amount'].describe())

### 4. Correlation Analysis
Let's look at numerical correlations. High correlation between variables can indicate multicollinearity.

In [ ]:
if 'df' in locals():
    # Select only numeric columns
    numeric_cols = df_filtered.select_dtypes(include=[np.number])
    
    plt.figure(figsize=(12, 8))
    corr = numeric_cols.corr()
    
    # Create a mask for the upper triangle
    mask = np.triu(np.ones_like(corr, dtype=bool))
    
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
    plt.title('Correlation Heatmap (TRANSFER and CASH_OUT only)')
    plt.show()

### 5. Time Behavior
Does fraud occur more frequently at specific times? (Step = 1 hour in PaySim)

In [ ]:
if 'df' in locals():
    plt.figure(figsize=(12, 6))
    sns.histplot(data=df, x='step', hue='isFraud', bins=100, element='step', common_norm=False, log_scale=(False, True))
    plt.title('Transaction Count over Time (Step = 1 hour)')
    plt.ylabel('Count (Log Scale)')
    plt.show()

### 6. Feature Distributions
Let's visualize the distributions of balance features for legitimate vs fraudulent transactions.

In [ ]:
if 'df' in locals():
    plt.figure(figsize=(15, 10))
    features_to_plot = ['oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
    for i, feature in enumerate(features_to_plot, 1):
        plt.subplot(2, 2, i)
        sns.boxplot(data=df_filtered, x='isFraud', y=feature)
        plt.title(f'{feature} by Fraud Status')
        plt.yscale('symlog')
    plt.tight_layout()
    plt.show()

### 7. Missingness Analysis
Are there any missing values in our dataset?

In [ ]:
if 'df' in locals():
    missing_data = df.isnull().sum()
    print("Missing values per column:")
    print(missing_data[missing_data > 0] if missing_data.sum() > 0 else "No missing values found.")

### 8. Outliers Analysis
Checking for amount outliers using IQR.

In [ ]:
if 'df' in locals():
    Q1 = df_filtered['amount'].quantile(0.25)
    Q3 = df_filtered['amount'].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 1.5 * IQR
    outliers = df_filtered[df_filtered['amount'] > upper_bound]
    print(f"Number of outlier transactions by amount (> {upper_bound:.2f}): {len(outliers):,}")
    fraud_in_outliers = outliers['isFraud'].sum()
    print(f"Fraudulent transactions among outliers: {fraud_in_outliers:,} ({(fraud_in_outliers/len(outliers))*100:.2f}%)")

### 9. Overall Class Distribution
Let's confirm the overall class imbalance.

In [ ]:
if 'df' in locals():
    class_dist = df['isFraud'].value_counts()
    print("Overall Class Distribution:")
    display(class_dist)
    print(f"Fraud Percentage: {(class_dist.get(1, 0) / len(df)) * 100:.4f}%")